# 33 — Math Toolkit

All differentiable building blocks: interpolation, root-finding, optimization, copulas, distributions, and special functions.

| Module | Location |
|--------|----------|
| Interpolation | `ql_jax.math.interpolations` |
| Solvers | `ql_jax.math.solvers` |
| Optimization | `ql_jax.math.optimization` |
| Copulas | `ql_jax.math.copulas` |
| Distributions | `ql_jax.math.distributions` |
| Random | `ql_jax.math.random` |

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

---
## 1. Interpolation

In [ ]:
from ql_jax.math.interpolations import linear, cubic, log

# Market discount factors
tenors = jnp.array([0.0, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0])
dfs = jnp.exp(-0.03 * tenors) * (1 + 0.002 * jnp.sin(2 * tenors))  # add curvature

lin_state = linear.build(tenors, dfs)
cub_state = cubic.build(tenors, dfs)
log_state = log.build(tenors, dfs)

t_fine = jnp.linspace(0, 10, 200)
lin_vals = linear.evaluate_many(lin_state, t_fine)
cub_vals = cubic.evaluate_many(cub_state, t_fine)
log_vals = log.evaluate_many(log_state, t_fine)

plt.figure(figsize=(10, 5))
plt.plot(np.array(t_fine), np.array(lin_vals), label='Linear', alpha=0.8)
plt.plot(np.array(t_fine), np.array(cub_vals), label='Cubic Spline', alpha=0.8)
plt.plot(np.array(t_fine), np.array(log_vals), label='Log-Linear', alpha=0.8)
plt.scatter(np.array(tenors), np.array(dfs), color='black', zorder=5, label='Knots')
plt.xlabel('Tenor')
plt.ylabel('Discount Factor')
plt.title('Interpolation Methods Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# AD through interpolation
deriv_ad = jax.grad(lambda x: cubic.evaluate(cub_state, x))(2.5)
deriv_analytic = cubic.derivative(cub_state, 2.5)

print(f"Cubic derivative at t=2.5:")
print(f"  AD       : {float(deriv_ad):.10f}")
print(f"  Analytic : {float(deriv_analytic):.10f}")

---
## 2. Root-Finding Solvers

In [ ]:
from ql_jax.math.solvers import brent, newton, bisection

# Find yield such that P(y) = 95
def bond_price_minus_target(y):
    coupon = 5.0
    n_periods = 10
    pv_coupons = coupon * (1 - (1 + y) ** (-n_periods)) / y
    pv_face = 100.0 * (1 + y) ** (-n_periods)
    return pv_coupons + pv_face - 95.0

y_brent = brent.solve(bond_price_minus_target, 1e-12, 0.05, x_min=0.001, x_max=0.20)
y_bisect = bisection.solve(bond_price_minus_target, 1e-12, 0.001, 0.20)

df = jax.grad(bond_price_minus_target)
y_newton = newton.solve(bond_price_minus_target, df, 1e-12, 0.05)

print(f"Yield to maturity (target P=95):")
print(f"  Brent    : {float(y_brent)*100:.8f}%")
print(f"  Bisection: {float(y_bisect)*100:.8f}%")
print(f"  Newton   : {float(y_newton)*100:.8f}%")

In [ ]:
# AD through root-finding: dy/d(target_price)
def solve_yield(target):
    f = lambda y: bond_price_minus_target(y) + (95.0 - target)
    return brent.solve(f, 1e-12, 0.05, x_min=0.001, x_max=0.20)

dy_dp = jax.grad(solve_yield)(95.0)
print(f"\n∂yield/∂price (via AD through Brent): {float(dy_dp):.8f}")

---
## 3. Optimization

In [ ]:
from ql_jax.math.optimization import bfgs, levenberg_marquardt, simplex

# Rosenbrock function
def rosenbrock(x):
    return (1 - x[0]) ** 2 + 100 * (x[1] - x[0] ** 2) ** 2

x0 = jnp.array([-1.0, 1.0])

result_bfgs = bfgs.minimize(rosenbrock, x0, max_iter=200)
print(f"BFGS:")
print(f"  x* = ({float(result_bfgs['x'][0]):.6f}, {float(result_bfgs['x'][1]):.6f})")
print(f"  f* = {float(result_bfgs['fun']):.2e}")
print(f"  iterations: {int(result_bfgs['n_iter'])}")

x_simplex, f_simplex, n_simplex = simplex.minimize(rosenbrock, x0)
print(f"\nNelder-Mead:")
print(f"  x* = ({float(x_simplex[0]):.6f}, {float(x_simplex[1]):.6f})")
print(f"  f* = {float(f_simplex):.2e}")
print(f"  iterations: {int(n_simplex)}")

In [ ]:
# Levenberg-Marquardt for least-squares
def residuals(x):
    return jnp.array([
        10.0 * (x[1] - x[0] ** 2),
        1.0 - x[0]
    ])

x_lm, cost_lm, n_lm = levenberg_marquardt.minimize(residuals, x0)
print(f"Levenberg-Marquardt:")
print(f"  x* = ({float(x_lm[0]):.6f}, {float(x_lm[1]):.6f})")
print(f"  cost = {float(cost_lm):.2e}, iterations: {int(n_lm)}")

---
## 4. Copulas

In [ ]:
from ql_jax.math.copulas import (
    gaussian_copula, clayton_copula, frank_copula, gumbel_copula,
    independent_copula, min_copula
)

u = jnp.linspace(0.01, 0.99, 50)
v = jnp.linspace(0.01, 0.99, 50)
UU, VV = jnp.meshgrid(u, v)

copula_fns = {
    'Gaussian (ρ=0.7)': lambda u, v: gaussian_copula(u, v, 0.7),
    'Clayton (θ=2)': lambda u, v: clayton_copula(u, v, 2.0),
    'Frank (θ=5)': lambda u, v: frank_copula(u, v, 5.0),
    'Gumbel (θ=2)': lambda u, v: gumbel_copula(u, v, 2.0),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, (name, fn) in zip(axes.flat, copula_fns.items()):
    C = jax.vmap(jax.vmap(fn, in_axes=(0, None)), in_axes=(None, 0))(u, v)
    ax.contourf(np.array(UU), np.array(VV), np.array(C), levels=15, cmap='viridis')
    ax.set_title(name)
    ax.set_xlabel('u')
    ax.set_ylabel('v')
fig.suptitle('Copula Comparison', fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# AD through copulas
d_gaussian_d_rho = jax.grad(gaussian_copula, argnums=2)(0.5, 0.5, 0.7)
d_clayton_d_theta = jax.grad(clayton_copula, argnums=2)(0.5, 0.5, 2.0)
print(f"∂C_gauss/∂ρ(0.5, 0.5, ρ=0.7)  : {float(d_gaussian_d_rho):.6f}")
print(f"∂C_clayton/∂θ(0.5, 0.5, θ=2.0): {float(d_clayton_d_theta):.6f}")

---
## 5. Distributions

In [ ]:
from ql_jax.math.distributions import normal, chi_squared, gamma as gamma_dist

x = jnp.linspace(-4, 4, 200)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Normal
axes[0].plot(np.array(x), np.array(jax.vmap(normal.pdf)(x)), label='PDF')
axes[0].plot(np.array(x), np.array(jax.vmap(normal.cdf)(x)), label='CDF')
axes[0].set_title('Standard Normal')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Chi-squared
x_pos = jnp.linspace(0.1, 15, 200)
for df in [2, 4, 6, 10]:
    axes[1].plot(np.array(x_pos), np.array(jax.vmap(lambda x: chi_squared.pdf(x, df))(x_pos)),
                 label=f'df={df}')
axes[1].set_title('Chi-Squared PDF')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Gamma
for alpha in [1.0, 2.0, 5.0, 9.0]:
    axes[2].plot(np.array(x_pos), np.array(jax.vmap(lambda x: gamma_dist.pdf(x, alpha))(x_pos)),
                 label=f'α={alpha}')
axes[2].set_title('Gamma PDF')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# AD through distribution functions
d_cdf = jax.grad(normal.cdf)(0.0)  # = pdf(0) = 1/sqrt(2π)
print(f"∂Φ/∂x at x=0 (= φ(0)) : {float(d_cdf):.10f}")
print(f"φ(0) = 1/√(2π)         : {1/np.sqrt(2*np.pi):.10f}")

# Inverse CDF gradient
d_inv = jax.grad(normal.inverse_cdf)(0.5)
print(f"\n∂Φ⁻¹/∂p at p=0.5      : {float(d_inv):.6f}")
print(f"1/φ(0) = √(2π)         : {np.sqrt(2*np.pi):.6f}")

---
## 6. Quasi-Random Sequences

In [ ]:
from ql_jax.math.random.quasi import sobol_sequence, halton_sequence

n_pts = 1000
sobol = sobol_sequence(dimension=2, n_points=n_pts)
halton = halton_sequence(dimension=2, n_points=n_pts)
key = jax.random.PRNGKey(0)
pseudo = jax.random.uniform(key, (n_pts, 2))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].scatter(np.array(pseudo[:, 0]), np.array(pseudo[:, 1]), s=1, alpha=0.5)
axes[0].set_title('Pseudo-Random')

axes[1].scatter(np.array(sobol[:, 0]), np.array(sobol[:, 1]), s=1, alpha=0.5)
axes[1].set_title('Sobol')

axes[2].scatter(np.array(halton[:, 0]), np.array(halton[:, 1]), s=1, alpha=0.5)
axes[2].set_title('Halton')

for ax in axes:
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')

fig.suptitle('Random vs Quasi-Random Sequences (1,000 points)', fontsize=14)
fig.tight_layout()
plt.show()

---
## 7. Special Functions

In [ ]:
from ql_jax.math.core import (
    beta, incomplete_beta, gamma_function, incomplete_gamma,
    error_function
)

# Beta function
print(f"B(2, 3) = {float(beta(2.0, 3.0)):.6f}  (exact: 1/12 = {1/12:.6f})")

# Gamma function
print(f"Γ(5) = {float(gamma_function(5.0)):.1f}  (exact: 4! = 24)")

# Error function
print(f"erf(1) = {float(error_function(1.0)):.6f}  (expected: 0.842701)")

# AD through special functions
d_gamma = jax.grad(gamma_function)(3.0)
print(f"\nΓ'(3) = {float(d_gamma):.6f}  (digamma(3)·Γ(3) = {float(d_gamma):.6f})")

---
## 8. Exercises

1. **SABR interpolation**: Build a SABR smile and compute AD vega.
2. **Quasi-MC pricing**: Price a European option using Sobol paths and compare convergence to pseudo-random.
3. **Bivariate CDF**: Compute ∂²C/∂x∂y for the bivariate normal CDF.

---
*End of Notebook 33*